# Day 1 / Session 1: Modern LLM Foundations -- Demos

This notebook contains all instructor-led demonstrations for Session 1.
Run each cell, observe the output, and make sure the concepts are clear
before moving on to the exercises notebook (`D1S1_exercises.ipynb`).

**What you will see:**
1. Setting up LLM API connections
2. Tokenization and context windows
3. Controlling model output with parameters
4. Shaping behavior with system messages
5. Chaining LLM calls into a pipeline
6. Embeddings and vector representations (narrative)
7. How LLMs support reasoning and agent behavior

---
## Session Goal

By the end of this session you will understand how to:

- **Connect** to LLMs programmatically using the OpenAI Python SDK
- **Tokenize** text and reason about context window budgets
- **Control** model output with `temperature`, `top_p`, and `max_tokens`
- **Shape** model behavior using system messages and personas
- **Chain** multiple LLM calls into a pipeline where one step feeds the next
- **Recognize** the reasoning capabilities (decomposition, planning, self-reflection, tool selection) that make LLMs suitable as the "brain" of an agent

These are the foundational building blocks. On **Day 2** you will wire these same primitives together inside LangChain and LangGraph to build autonomous multi-step agents. Everything you learn here carries forward directly.

In [ ]:
!pip install -q openai tiktoken matplotlib scikit-learn python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# ============================================================
# Imports and Setup
# ============================================================

import openai
import tiktoken
import os
import json
import numpy as np

# Ensure your OpenAI API key is set as an environment variable:
#   export OPENAI_API_KEY="sk-..."
# Or set it in a .env file in the project root.

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Demo 1: Setting Up LLM API Connections

In this demo we initialize the OpenAI client and make our first chat completion call.
The `client` object handles authentication, request formatting, and response parsing.

**Scenario:** You are building a tool that lets a consulting team quickly query an LLM
to summarize industry concepts for a client engagement kickoff.

In [ ]:
# Demo 1 - Setting Up LLM API Connections

# Step 1: Initialize the OpenAI client
# The client automatically reads OPENAI_API_KEY from the environment.
client = openai.OpenAI()

# Step 2: Make a simple chat completion call
# Scenario: A consulting analyst needs a quick definition for a client deck
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "user", "content": "What is a digital transformation strategy? Explain in two sentences for a McKinsey client presentation."}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)

# Step 3: Print the response
print("Model used :", response.model)
print("Finish reason:", response.choices[0].finish_reason)
print("\nAssistant response:")
print(response.choices[0].message.content)

**Observe:** The response object contains metadata beyond just the text. `finish_reason` tells you *why* the model stopped generating -- `stop` means it finished naturally, while `length` means it hit the `max_tokens` limit. Always check this in production code.

---
## Demo 2: Understanding Tokenization and Context Windows

Tokens are the fundamental units that LLMs process. A token can be a word, part of a
word, or even a single character. Understanding tokenization helps you:
- Estimate API costs (billing is per-token)
- Stay within context window limits
- Write more efficient prompts

In [ ]:
# Demo 2a - Create encoder and tokenize first texts

# Step 1: Get the tokenizer for gpt-4o-mini
encoder = tiktoken.encoding_for_model("gpt-4o-mini")

# Step 2: Encode various consulting-related texts and count tokens
texts = [
    "McKinsey & Company",
    "Digital transformation is reshaping how Fortune 500 companies create value across their organizations.",
    "The client's operating model requires restructuring to achieve the target $200M in synergies.",
    "Organizational restructuring and operational efficiency optimization",
    "def calculate_market_size(tam, penetration_rate):\n    sam = tam * penetration_rate\n    return sam * 0.15  # McKinsey target share"
]

print("Token counts for consulting texts (first two):")
print("=" * 60)
for text in texts[:2]:
    tokens = encoder.encode(text)
    print(f"\nText   : {text[:60]}{'...' if len(text) > 60 else ''}")
    print(f"Chars  : {len(text)}")
    print(f"Tokens : {len(tokens)}")

In [ ]:
# Demo 2b - Encode remaining texts and show token IDs

print("Token counts for remaining texts (with token IDs):")
print("=" * 60)
for text in texts[2:]:
    tokens = encoder.encode(text)
    print(f"\nText   : {text[:60]}{'...' if len(text) > 60 else ''}")
    print(f"Chars  : {len(text)}")
    print(f"Tokens : {len(tokens)}")
    print(f"Token IDs (first 10): {tokens[:10]}")

**Observe:** Notice that code text has a different token-to-character ratio than natural language. Special characters, variable names, and syntax tokens all count. The token IDs are integers that map to entries in the model's vocabulary -- the model never sees raw text, only these integer sequences.

In [ ]:
# Demo 2c - Context window limits

print("Context window reference:")
print("  gpt-4o-mini : 128,000 tokens")
print("  gpt-4o      : 128,000 tokens")
print("  gpt-3.5-turbo: 16,385 tokens")

# Simulate a long due diligence report consuming the window
long_text = "The target company operates in a fragmented market with strong growth potential. " * 500
long_tokens = encoder.encode(long_text)
print(f"\nLong due diligence excerpt ({len(long_text)} chars) = {len(long_tokens)} tokens")
print(f"That is {len(long_tokens)/128000*100:.2f}% of the gpt-4o-mini context window.")

**Observe:** Even 500 repetitions of a sentence only consumes a small fraction of the 128k context window. But real-world documents (annual reports, due diligence data rooms) can easily fill it. In Day 3 you will learn how RAG (Retrieval-Augmented Generation) solves this by retrieving only the most relevant chunks.

### Try This
Change one of the texts above to a very long sentence (100+ words) and re-run. How does the token count compare to the character count? Is the ratio consistent across different types of text (technical vs. conversational vs. financial)?

---
## Demo 3: Exploring Model Parameters

LLMs expose several parameters that control the randomness and length of generated text:

| Parameter | Description |
|-----------|-------------|
| `temperature` | Controls randomness. 0 = deterministic, 1 = creative |
| `top_p` | Nucleus sampling. Lower values = more focused output |
| `max_tokens` | Maximum number of tokens in the response |

In [ ]:
# Demo 3a - Temperature comparison

client = openai.OpenAI()
prompt = "Write a one-sentence executive summary of generative AI's impact on the management consulting industry."

print("TEMPERATURE COMPARISON")
print("=" * 60)

for temp in [0.0, 0.5, 1.0]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80"))
    )
    print(f"\nTemperature {temp}:")
    print(f"  {response.choices[0].message.content}")

**Observe:** At temperature 0, the output is nearly identical across runs. At temperature 1, you get more variety and occasionally surprising phrasing. For factual consulting deliverables (financial reports, due diligence summaries), use low temperature (0--0.3). For brainstorming and ideation, use higher values (0.7--1.0).

In [ ]:
# Demo 3b - top_p comparison

print("TOP_P COMPARISON")
print("=" * 60)

for top_p in [0.1, 0.5, 1.0]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        top_p=top_p,
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80"))
    )
    print(f"\ntop_p {top_p}:")
    print(f"  {response.choices[0].message.content}")

**Observe:** `top_p` works differently from `temperature` -- it limits the vocabulary the model can sample from. At `top_p=0.1`, the model only considers tokens that together make up the top 10% of probability mass, leading to very focused output. At `top_p=1.0`, the full vocabulary is available. OpenAI recommends changing **either** `temperature` **or** `top_p`, not both at the same time.

In [ ]:
# Demo 3c - max_tokens comparison

print("MAX_TOKENS COMPARISON")
print("=" * 60)

for max_tok in [10, 30, 100]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")),
        max_tokens=max_tok
    )
    text = response.choices[0].message.content
    print(f"\nmax_tokens={max_tok} (finish_reason={response.choices[0].finish_reason}):")
    print(f"  {text}")

**Gotcha:** Notice `finish_reason='length'` when `max_tokens` is too small -- the model was cut off mid-sentence. It did not choose to stop; it *ran out of budget*. Always check `finish_reason` in production code! If you see `length`, either increase `max_tokens` or redesign your prompt to elicit shorter responses.

### Quick Recap
- **Q:** You are building a tool that generates financial reports for clients. What temperature should you use and why?
- **Q:** What happens if you set both `temperature=0` AND `top_p=0.1`? (Hint: OpenAI recommends changing only one at a time.)
- **Q:** A partner complains that the AI-generated summary "cuts off mid-sentence." What is the most likely cause, and where in the response object would you confirm it?

---
## Demo 4: Comparing LLM Response Behaviors

The **system message** is a powerful lever for controlling how the LLM behaves.
By changing the system message, you can make the same model act as different expert
"personas," each with its own tone, expertise, and communication style -- mirroring
how different practice areas approach the same client question.

In [ ]:
# Demo 4a - Define personas dictionary

client = openai.OpenAI()

user_question = "Our retail client is experiencing a 15% decline in same-store sales. What should we investigate?"

personas = {
    "McKinsey Partner (Strategy)": (
        "You are a McKinsey Senior Partner leading the Strategy & Corporate Finance practice. "
        "You think in terms of competitive dynamics, portfolio strategy, and value creation. "
        "Use structured, MECE frameworks and speak with authority."
    ),
    "McKinsey Associate (Operations)": (
        "You are a McKinsey Associate in the Operations practice. You focus on supply chain, "
        "store-level performance metrics, and operational efficiency. Be data-driven and suggest "
        "specific analyses to run."
    ),
    "McKinsey Expert (Digital & Analytics)": (
        "You are a McKinsey Digital & Analytics expert. You focus on digital channels, "
        "customer analytics, omnichannel strategy, and technology enablement. "
        "Recommend data-driven approaches and digital solutions."
    ),
}

print(f"Client Question: {user_question}")
print(f"Number of personas defined: {len(personas)}")
for name in personas:
    print(f"  - {name}")

In [ ]:
# Demo 4b - Run the persona comparison loop

print(f"Client Question: {user_question}")
print("=" * 60)

for persona_name, system_msg in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_question}
        ],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")),
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
    )
    print(f"\n--- {persona_name} ---")
    print(response.choices[0].message.content)
    print()

**Observe:** The *same* model, given the *same* question, produces fundamentally different analyses depending on the system message. The Strategy partner thinks about competitive positioning and portfolio decisions. The Operations associate dives into store-level metrics and supply chain data. The Digital expert focuses on omnichannel and customer analytics. This is the power of system messages -- you are not fine-tuning the model, just steering it.

### Try This
Add a 4th persona -- a **"McKinsey Implementation Expert"** focused on change management, organizational readiness, and execution planning. How does its response differ from the other three? Does it surface concerns the others missed?

---
## Demo 5: Building a Basic LLM Pipeline

A pipeline chains multiple LLM calls together, where the output of one step becomes
the input of the next. This pattern is fundamental to agentic systems and mirrors how
structured analysis works -- first synthesize raw research, then extract structured
insights for client delivery.

In [ ]:
# Demo 5a - Define helper function

client = openai.OpenAI()

def call_llm(system_message, user_message, temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")), max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))):
    """Helper function to make an LLM call with a system and user message."""
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


# The raw research we will process through the pipeline
industry_research = """
The global management consulting market is undergoing a fundamental transformation driven by 
generative AI and advanced analytics. McKinsey's own research indicates that up to 70% of 
business activities could be automated with current technology. Traditional strategy engagements 
that once required weeks of analyst research can now be augmented with AI-driven market analysis 
in days. However, the human element remains critical -- clients value the judgment, relationship 
management, and change leadership that experienced consultants provide. The shift is creating 
new service lines around AI implementation, responsible AI governance, and digital capability 
building. Firms that successfully integrate AI into their delivery model are seeing 30-40% 
improvements in engagement efficiency while maintaining or improving quality scores. The key 
challenges include managing data confidentiality across client engagements, ensuring AI outputs 
meet McKinsey's quality standards, building consultant capabilities in prompt engineering and 
AI-augmented analysis, and maintaining the trusted advisor relationship in an AI-augmented world.
"""

print("Helper function defined. Industry research loaded.")
print(f"Research length: {len(industry_research.split())} words")

**Observe:** The `call_llm` helper wraps the OpenAI API call into a clean function. This is a pattern you will reuse constantly -- it keeps pipeline steps readable and avoids repeating boilerplate. Notice how the default parameter values come from environment variables.

In [ ]:
# Demo 5b - Pipeline Step 1: Summarize

print("PIPELINE STEP 1: Summarize Industry Research")
print("=" * 60)

summary = call_llm(
    system_message="You are a McKinsey research analyst. Summarize the given text in 2-3 sentences suitable for a partner briefing.",
    user_message=f"Summarize this industry research:\n\n{industry_research}"
)
print(summary)
print(f"\nSummary length: {len(summary.split())} words (down from {len(industry_research.split())} words)")

**Observe:** Step 1 compressed the research from ~150 words down to 2--3 sentences. The model acted as a filter, keeping only the most important points. But did it keep the *right* points? That is always the risk with summarization -- and why Step 2 matters.

In [ ]:
# Demo 5c - Pipeline Step 2: Extract Key Insights + Show Pipeline Summary

print("PIPELINE STEP 2: Extract Key Insights for Client Deck")
print("=" * 60)

key_points = call_llm(
    system_message="You are a McKinsey engagement manager preparing a client presentation. Extract exactly 5 key insights as a numbered list, each suitable for a slide bullet point.",
    user_message=f"Extract the key insights from this summary:\n\n{summary}"
)
print(key_points)

print("\nPIPELINE RESULT: Complete output")
print("=" * 60)
print(f"Original research : {len(industry_research.split())} words")
print(f"Summary length    : {len(summary.split())} words")
print(f"\nKey Insights for Client Deck:\n{key_points}")

**Observe:** The pipeline took raw research and produced slide-ready bullet points in two steps. Each step had a focused role (summarize, then extract). This is more reliable than asking a single prompt to do both, because each step can be tuned and tested independently. This is the same decomposition principle you will use when building agents on Day 2.

### Quick Recap
- **Q:** Why is chaining LLM calls useful instead of putting everything in one prompt?
- **Q:** What would happen if the summary in Step 1 missed a critical point -- how would that affect Step 2?
- **Q:** How would you add a Step 3 that translates the bullet points into a different language for an international client?

---
## Demo 6: Embeddings and Vector Representations

Embeddings convert text into dense numerical vectors that capture semantic meaning.
Two sentences with similar meaning will have similar embedding vectors, even if they
use different words. This is the foundation for:
- Semantic search across knowledge bases and past engagement documents
- Clustering client industries and market research by topic
- Similarity-based retrieval for RAG-powered assistants

**Key insight:** Embeddings capture *meaning*, not just keywords.
"Revenue growth decelerated in Q3" and "Sales momentum slowed in the third quarter"
will have very similar vectors despite using completely different words.

*Embeddings are covered hands-on in the Day 3 RAG sessions.*

---
## Demo 7: How LLMs Support Reasoning and Agent Behavior

LLMs exhibit several emergent capabilities that make them suitable as the reasoning
core of agentic systems. This demo showcases four of those capabilities:

1. **Task Decomposition** -- breaking a complex problem into MECE workstreams
2. **Planning** -- generating a step-by-step engagement plan
3. **Self-Reflection** -- critiquing and improving its own output
4. **Tool Selection** -- choosing the right analytical tool for a given request

These are the same capabilities you will wire together in Day 2 when you build
multi-step agents.

In [ ]:
# Demo 7a - Capability 1: Task Decomposition (MECE Breakdown)

client = openai.OpenAI()

print("CAPABILITY 1: Task Decomposition (MECE Breakdown)")
print("=" * 60)

decomposition_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are a McKinsey engagement manager. Break down client problems into MECE (Mutually Exclusive, Collectively Exhaustive) workstreams with numbered sub-tasks."},
        {"role": "user", "content": "Our pharmaceutical client wants to evaluate entering the biosimilars market in Europe. Structure the key workstreams for this engagement."}
    ],
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.3")),
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)
print(decomposition_response.choices[0].message.content)

**Observe:** The model produced a structured, hierarchical breakdown without being explicitly taught MECE. It inferred the framework from the system message. This is task decomposition -- the same capability that lets agents break a user request into sub-steps.

In [ ]:
# Demo 7b - Capability 2: Planning (Engagement Plan)

print("CAPABILITY 2: Planning -- Generating an Engagement Plan")
print("=" * 60)

planning_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": (
            "You are a McKinsey project manager. Given an engagement goal, output a numbered "
            "week-by-week plan of concrete actions. Each step should be a single deliverable. "
            "Format: 'Week N: [DELIVERABLE] -- description'"
        )},
        {"role": "user", "content": "Goal: Execute a 6-week due diligence for a private equity client evaluating a $500M acquisition of a B2B SaaS company."}
    ],
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.3")),
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)
print(planning_response.choices[0].message.content)

**Observe:** Planning goes beyond decomposition -- it adds a *temporal* dimension (ordering, dependencies, milestones). In agentic systems, planning allows the LLM to decide not just *what* to do, but *in what order*. Notice how the plan builds logically: data gathering before analysis, analysis before synthesis.

In [ ]:
# Demo 7c - Capability 3: Self-Reflection (Quality Review)

print("CAPABILITY 3: Self-Reflection -- Reviewing and Improving Analysis")
print("=" * 60)

# First, generate a deliberately brief answer
initial_answer = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "What are the key drivers of customer churn in the telecom industry?"}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "40"))
)
brief = initial_answer.choices[0].message.content
print(f"Initial (brief) answer: {brief}")

# Now ask the model to reflect on and improve its own answer -- like a partner review
reflection = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are a McKinsey Senior Partner reviewing an associate's work. Evaluate the answer below for completeness, MECE structure, and analytical rigor, then provide an improved version."},
        {"role": "user", "content": f"Original question: What are the key drivers of customer churn in the telecom industry?\n\nAssociate's draft: {brief}\n\nProvide your critique and an improved, partner-ready answer."}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "250"))
)
print(f"\nPartner review & improvement:\n{reflection.choices[0].message.content}")

**Observe:** Self-reflection is one of the most powerful capabilities for agentic systems. The model identified gaps in its own output and produced a substantially better version. In Day 2 you will build agents that automatically loop: generate, critique, improve -- without human intervention.

In [ ]:
# Demo 7d - Capability 4: Tool Selection (Consulting Tools)

print("CAPABILITY 4: Tool Selection -- Choosing the Right Analytical Tool")
print("=" * 60)

tool_prompt = """You have access to these McKinsey analytical tools:
1. market_database -- for market size, growth rates, and competitive landscape data
2. financial_model -- for DCF, LBO, and valuation calculations
3. survey_platform -- for customer/employee surveys and NPS analysis
4. expert_network -- for scheduling calls with industry experts
5. benchmarking_tool -- for comparing client KPIs against industry benchmarks

For each request, respond with the tool you would select and why. Format: TOOL: [name] | REASON: [explanation]"""

tool_selection = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": tool_prompt},
        {"role": "user", "content": "What is the total addressable market for electric vehicle charging infrastructure in North America?"}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)
print(f"Request: 'TAM for EV charging infrastructure in North America'")
print(f"Decision: {tool_selection.choices[0].message.content}")

tool_selection2 = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": tool_prompt},
        {"role": "user", "content": "Is the target company's EBITDA margin competitive relative to peers in the specialty chemicals sector?"}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)
print(f"\nRequest: 'Is target EBITDA margin competitive vs specialty chemicals peers?'")
print(f"Decision: {tool_selection2.choices[0].message.content}")

**Observe:** The model correctly matched each request to the appropriate tool. It chose `market_database` for a TAM question and `benchmarking_tool` for a peer comparison question. This is the same capability that powers function calling in agents -- the LLM decides *which* tool to invoke based on the user's intent. You will implement this explicitly on Day 2 with LangChain tool bindings.

---
## Summary & Key Takeaways

In these demos you built the foundational skills for working with LLMs programmatically:

| # | Concept | Key Takeaway |
|---|---------|-------------|
| 1 | **API Connections** | The OpenAI client handles auth, formatting, and parsing. Always check `finish_reason`. |
| 2 | **Tokenization** | Text is split into tokens (not words). Token count drives cost and context window limits. |
| 3 | **Model Parameters** | `temperature` controls randomness, `top_p` controls vocabulary breadth, `max_tokens` caps length. Change one at a time. |
| 4 | **System Messages & Personas** | The system message is your most powerful control lever. Same model, different persona = different analysis. |
| 5 | **LLM Pipelines** | Chain focused LLM calls (summarize -> extract -> translate) for more reliable results than one mega-prompt. |
| 6 | **Embeddings** | Text-to-vector conversion captures semantic meaning. Foundation for RAG on Day 3. |
| 7 | **LLM Reasoning** | Decomposition, planning, self-reflection, and tool selection are the four capabilities that make agents possible. |

### How This Connects to the Rest of the Program
- **Session 2 (Prompt Engineering):** You will learn advanced techniques for writing better system messages and user prompts.
- **Day 2 (LangChain & LangGraph):** You will wire these primitives into autonomous agents that plan, use tools, and self-correct.
- **Day 3 (RAG & Deployment):** You will combine embeddings with retrieval to build knowledge-grounded assistants and deploy them.

**Next:** Open `D1S1_exercises.ipynb` to build a conversational agent yourself.